# nil-xit Jupyter Integration

## Setup And Imports

In [ ]:
import os
import json
import threading

import nil_service
import nil_xit

port = 1101

## Runtime Helpers

In [ ]:
class JSONValue:
    """
    Convenience wrapper for JSON-encoded frame values.

    nil-xit itself is serialization-agnostic.
    This helper simply layers JSON encoding on top
    of the raw payload synchronization API.
    """
    def __init__(self, frame: nil_xit.UniqueFrame, id: str, data):
        self.data= json.dumps(data).encode()
        self.frame = frame

        def get():
            return self.data

        def set(data):
            print(f"Received data: {data.decode()}")
            self.data = data
        
        self.value = self.frame.add_value(id, get, set)

from IPython.display import HTML

def view_frames(frames: list[str], template: list[list[int]]):
    return HTML(f"""
    <div class="root">
        {"\n".join(f'<iframe style="grid-area: {frame}" src="http://127.0.0.1:{port}/?frame={frame}"></iframe>' for frame in frames)}
    </div>
    <style>
        .root {{
            display: grid;
            width: 100%;
            height: {len(template) * 400}px;
            grid-template-areas:
                {"\n".join(f'"{" ".join(frames[l] for l in line)}"' for line in template)};
            gap: 16px;
        }}
        iframe {{
            width: 100%;
            height: 100%;
            border: 1px solid #ccc;
        }}
    </style>
    """)

## Frame Definitions

In [ ]:
def create_index_frame(xit: nil_xit.Core):
    index_frame = xit.add_unique_frame("index", nil_xit.FileInfo("local", "Main.svelte"))
    index_frame.add_signal("click", lambda data: print(f"clicked: {data.decode()}"))
    return index_frame

def create_data_frame(xit: nil_xit.Core):
    frame = xit.add_unique_frame("data")
    value = JSONValue(frame, "data", [{
        "x": ["Apples", "Bananas", "Cherries"],
        "y": [10, 15, 8],
        "type": "bar",
        "marker": { "color": 'rgb(99, 255, 132)' }
    }])

    return frame, value

def create_plotly_frame(xit: nil_xit.Core):
    frame = xit.add_unique_frame("plotly", nil_xit.FileInfo("local", "SharedComponent.svelte"))
    frame.add_option("component", "@nil-/xit/components/3pt/Plotly.svelte")
    return frame

def create_json_editor_frame(xit: nil_xit.Core):
    frame = xit.add_unique_frame("json_editor", nil_xit.FileInfo("local", "SharedComponent.svelte"))
    frame.add_option("component", "@nil-/xit/components/3pt/JSONEditor.svelte")
    return frame

def create_desmos_frame(xit: nil_xit.Core):
    frame = xit.add_unique_frame("desmos", nil_xit.FileInfo("local", "Component.svelte"))
    frame.add_option("component", "@nil-/xit/components/3pt/Desmos.svelte")
    value = JSONValue(frame, "data", [{ "id": "1", "type": "expression", "latex": "x = y / 2" }])
    return frame, value

## Server Initialization

In [ ]:
server = nil_service.create_http_server("127.0.0.1", port, 100 * 1024 * 1024)
nil_xit.setup_server(server)

server.on_ready(lambda id: print(f"http://{id.to_string()}"))

xit = nil_xit.create_core(server, server.use_ws("/ws"))
xit.set_groups({ "local": f"{os.getcwd()}/../app/gui/local" })

index = create_index_frame(xit)
data = create_data_frame(xit)
plotly = create_plotly_frame(xit)
json_editor = create_json_editor_frame(xit)
desmos = create_desmos_frame(xit)

stop_event = threading.Event()

def poll_server():
    while not stop_event.is_set():
        server.poll()

poll_thread = threading.Thread(target=poll_server, daemon=True)
poll_thread.start()

## Live UI Embedding

In [ ]:
view_frames(
    frames=["plotly", "json_editor", "desmos"],
    template=[
        [0, 1],
        [2, 2]
    ]
)